**Required imports**

In [1]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import numpy as np
import joblib as jb

**Loading Dataset into pandas dataframe from hugging face**

In [2]:
ds = load_dataset("milistu/AMAZON-Products-2023")
x_train=pd.DataFrame(ds['train'].select_columns(['title', 'description', 'main_category']))
x_train.head()
x_train.to_csv('Real_dataset.csv',index=False)

**Analyzing Data**

In [3]:
print(f"The number of rows and columns are : {x_train.shape}")
print(f"There are {x_train['main_category'].nunique()} unique categories")
print(f"\nNull values\n{x_train.isnull().sum()}")

The number of rows and columns are : (117243, 3)
There are 37 unique categories

Null values
title                0
description          0
main_category    24805
dtype: int64


lol 24k null values in main_category feature

**predicting those 24k null values in categories_prediction.ipynb**

In [4]:
print(f"\n{x_train['main_category'].value_counts().head(20)}")


main_category
AMAZON FASHION               36460
Amazon Home                  13874
Tools & Home Improvement      7945
Automotive                    5980
Cell Phones & Accessories     3471
Health & Personal Care        3416
All Electronics               3136
Sports & Outdoors             2942
Office Products               2925
Industrial & Scientific       2460
Pet Supplies                  1907
Computers                     1724
Digital Music                 1379
Handmade                      1127
Arts, Crafts & Sewing          874
Camera & Photo                 842
Musical Instruments            506
Appliances                     312
Home Audio & Theater           218
Video Games                    182
Name: count, dtype: int64


In [5]:
print(f"\n{x_train['main_category'].value_counts().tail(18)}")


main_category
Video Games                     182
Toys & Games                    146
Appstore for Android            142
Collectibles & Fine Art         114
Car Electronics                  69
All Beauty                       65
GPS & Navigation                 54
Portable Audio & Accessories     32
Collectible Coins                31
Grocery                          24
Baby                             23
Sports Collectibles              21
Software                         12
Entertainment                    11
Gift Cards                        8
Magazine Subscriptions            3
Amazon Devices                    2
Amazon Fire TV                    1
Name: count, dtype: int64


**Loading main_category predicted cleaned data**

In [6]:
cleaned_df=pd.read_csv('clean_data.csv')
cleaned_df.head()

,title,description,main_category
0,anomie bonhomie,amazoncom fan of scritti polittis synthpopfunk...,digital music
1,sunshine on my shoulder the best of john denve...,sunshine on my shoulder be a 2cd 36track colle...,digital music
2,18 greatest hit of 38 special,track listings 1 rockin into the night 2 hold ...,digital music
3,the gift cd,second studio album by the multimillionselling...,digital music
4,τηε βοοτlεg sεrιεs vοι ᛐ7 ᛐ996ᛐ997 frαԍμεντտ τ...,eu edition 2cd disc one time out of mind 2022 ...,digital music


In [7]:
cleaned_df['main_category'].value_counts().tail(20)

main_category
all electronics            4278
office products            3219
industrial & scientific    2967
pet supplies               2723
computers                  2303
handmade                   1993
digital music              1410
arts, crafts & sewing      1310
camera & photo              890
musical instruments         528
appliances                  441
video games                 257
home audio & theater        231
other                       197
toys & games                195
appstore for android        150
collectibles & fine art     115
all beauty                   80
car electronics              76
gps & navigation             61
Name: count, dtype: int64

analyzing and cleaning null data

In [8]:
print(cleaned_df.isnull().sum())
cleaned_df=cleaned_df.dropna()
print(f"\n\nAfter droping\n\n{cleaned_df.isna().sum()}")

title             6
description      70
main_category     0
dtype: int64


After droping

title            0
description      0
main_category    0
dtype: int64


**converting the text into vector**

In [9]:
t_tfidf=TfidfVectorizer(max_features=25000, ngram_range=(1,2), stop_words='english')
c_tfidf=TfidfVectorizer(max_features=50,stop_words='english')
d_tfidf=TfidfVectorizer(max_features=25000,ngram_range=(1,2),stop_words='english')

In [10]:
t_vector=t_tfidf.fit_transform(cleaned_df['title'])
c_vector=c_tfidf.fit_transform(cleaned_df['main_category'])
d_vector=d_tfidf.fit_transform(cleaned_df['description'])

assigning weight to each column based on their importance

In [11]:
tw=3.0
cw=2.0
dw=1.0

In [12]:
t_vector=t_vector*tw
c_vector=c_vector*cw
d_vector=d_vector*dw

final entire vectorized sparse matrix

In [13]:
s_v=sp.hstack([t_vector,c_vector,d_vector],format='csr')

**Function to calculate and return top similar product of query**

In [14]:
def similarity_calculate(title="",category="",description="",top_similar=5):
    title=title.lower()
    category=category.lower()
    description=description.lower()
    title_vector=t_tfidf.transform([title])*tw
    category_vector=c_tfidf.transform([category])*cw
    description_vector=d_tfidf.transform([description])*dw
    final_vector=sp.hstack([title_vector,category_vector,description_vector],format='csr')
    similarity=cosine_similarity(final_vector,s_v).flatten()
    top_similar_indx=np.argsort(similarity)[::-1][:top_similar]
    recommended = cleaned_df.iloc[top_similar_indx].copy()
    return recommended[['title','main_category']]

**checking model output**

In [15]:
print(similarity_calculate("Nike shoes","fashion"))

                          title   main_category
76653      nike girls boyshorts  amazon fashion
80513               nike casual  amazon fashion
72717               nike modern  amazon fashion
109704             nike hoodies  amazon fashion
77047   nike unisexchild modern  amazon fashion


In [16]:
jb.dump(t_tfidf,"title.pkl")
jb.dump(c_tfidf,'category.pkl')
jb.dump(d_tfidf,'description.pkl')
jb.dump(s_v,'sparse_vectorized_matrix.pkl')
jb.dump(cleaned_df,'products.pkl')

['products.pkl']